# IZQAR IEEE-CIS training on Kaggle
Accept the competition rules yourself, then Add Input: `ieee-fraud-detection`. Enable Internet for the first cell to clone the public IZQAR feature branch. This notebook uses the repository's `backend/scripts/train_ieee.py` and only the labeled train CSV files.


In [ ]:
from pathlib import Path
import subprocess, sys
root = Path('/kaggle/working/izqar')
if not root.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'feat/ml-pipelines-and-startup', 'https://github.com/Night011/izqar.git', str(root)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(root / 'backend/requirements-ml.txt')], check=True)
tx = list(Path('/kaggle/input').rglob('train_transaction.csv'))
identity = list(Path('/kaggle/input').rglob('train_identity.csv'))
if len(tx) != 1 or len(identity) != 1 or tx[0].parent != identity[0].parent:
    raise FileNotFoundError('Add Input: ieee-fraud-detection; expected train_transaction.csv and train_identity.csv in one folder.')
print('IEEE-CIS inputs:', tx[0], identity[0])


In [ ]:
from pathlib import Path
import subprocess, sys
backend = root / 'backend'
experiment_reports = []
for count in (100000, 200000, 300000):
    output = Path(f'/kaggle/working/experiments/ieee_{count//1000}k.joblib')
    subprocess.run([sys.executable, str(backend / 'scripts/train_ieee.py'), '--data-dir', str(tx[0].parent), '--output', str(output), '--mode', 'experiment', '--max-train-rows', str(count), '--estimators', '180'], cwd=backend, check=True)
    experiment_reports.append(output.with_suffix('.metrics.json'))
print('Review validation reports, threshold curves and feature importance before final evaluation:', experiment_reports)


Review the three validation reports. Fix the threshold strategy and parameters before setting `RUN_FINAL = True`. The final cell evaluates the held-out test once.


In [ ]:
RUN_FINAL = False  # Set True only after reviewing validation reports.
if not RUN_FINAL:
    print('Final test has not been evaluated. Set RUN_FINAL=True when the protocol is fixed.')
else:
    from pathlib import Path
    import subprocess, sys, zipfile
    backend = root / 'backend'
    output = Path('/kaggle/working/ieee_model.joblib')
    subprocess.run([sys.executable, str(backend / 'scripts/train_ieee.py'), '--data-dir', str(tx[0].parent), '--output', str(output), '--estimators', '180', '--mode', 'final'], cwd=backend, check=True)
    report = output.with_suffix('.metrics.json')
    bundle = Path('/kaggle/working/izqar_ieee_artifacts.zip')
    with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(output, output.name)
        archive.write(report, report.name)
        for experiment_report in experiment_reports:
            archive.write(experiment_report, 'experiments/' + experiment_report.name)
    print('Download from Kaggle Output:', bundle)
